
### **Data Cleaning**
* Parse dates
* Handle zone and TemperatureSupport missingness
* Normalize binary/boolean cols
* Validity check
* Duplicate check (post-cleaning)
* Save cleaned dataset

In [1]:
#library imports
import pandas as pd
import numpy as np 

raw_path = "../data/raw/enhanced_smart_waste_dataset_v1.csv"
dataset = pd.read_csv(raw_path)
dataset.shape

(1000, 19)

#### Parsing Dates
* Dates in the raw file are strings, not ISO format

In [2]:
dataset["ArrivalDate"] = pd.to_datetime(dataset["ArrivalDate"], dayfirst=True)
dataset["ExpiryDate"]  = pd.to_datetime(dataset["ExpiryDate"], dayfirst=True)

bad_dates = dataset[dataset["ExpiryDate"] < dataset["ArrivalDate"]]

#### Handling missing Zone and TemperatureSupport 

**Extent of missingness**
Zone and TemperatureSupport are both missing in 656 of 1000 records ~65.6%; identical missingness pattern across both columns, which itself is a clue worth noting (we assume they're likely missing together for the same underlying reason, not independently).

**Standard imputation doesn't fit**
Mean/mode imputation assumes the data is closer to `MCAR`, that a small number of gaps can be filled with a "typical" value without distorting the dataset. At `65.6%` missing, that assumption breaks down. Imputing a single dominant zone of records wouldn't be filling a gap, it would be inventing the majority of the column's values outright.

**Dropping rows isn't viable either**
Dropping every row with a missing value would remove 656 of 1000 records, discarding most of the dataset to preserve two columns of secondary importance to the prediction target.
**chosen approach**
Both columns are recorded with an explicit "Unknown" category rather than imputed. This treats the absense of a value as a legitimate category not a placeholder guess. 

In [3]:
dataset["Zone_Missing"] = dataset["Zone"].isna().astype(int)
dataset["TemperatureSupport_Missing"] = dataset["TemperatureSupport"].isna().astype(int)

dataset["Zone"] = dataset["Zone"].fillna("Unknown")
dataset["TemperatureSupport"] = dataset["TemperatureSupport"].map({True: "Yes", False:"No"}).fillna("Unknown")
dataset[["Zone", "Zone_Missing", "TemperatureSupport", "TemperatureSupport_Missing"]].head(10)

,Zone,Zone_Missing,TemperatureSupport,TemperatureSupport_Missing
0,Unknown,1,Unknown,1
1,Unknown,1,Unknown,1
2,West,0,No,0
3,Unknown,1,Unknown,1
4,West,0,No,0
5,North,0,Yes,0
6,Unknown,1,Unknown,1
7,Unknown,1,Unknown,1
8,Unknown,1,Unknown,1
9,North,0,Yes,0


#### Normalizing binary/boolean cols

In [4]:
dataset["OnPromotion"] = dataset["OnPromotion"].map({"Yes": True, "No": False})
assert dataset["OnPromotion"].isna().sum() == 0, "Unpected value in Promotion"
dataset[["OnPromotion", "IsSpoiled", "TemperatureSensitive"]].dtypes

OnPromotion             bool
IsSpoiled               bool
TemperatureSensitive    bool
dtype: object

#### Validity checks on numeric cols

In [5]:
numeric_checks = ["StockQty", "DaysUntilExpiry", "DailySaleAvg", "DistanceToNearestStore", "AvgDailySaleInNearbyStores"]

for col in numeric_checks:
    n_negative = (dataset[col] < 0).sum()
    print(f"{col}: {n_negative} negative values")

StockQty: 0 negative values
DaysUntilExpiry: 0 negative values
DailySaleAvg: 0 negative values
DistanceToNearestStore: 0 negative values
AvgDailySaleInNearbyStores: 0 negative values


##### **Duplicate Check** (repeat, post-cleaning)


In [6]:
print("Duplicates rows after cleaning:", dataset.duplicated().sum())

Duplicates rows after cleaning: 0


##### Verifying, there are no missing values

In [7]:
assert dataset.isna().sum().sum()  ==  0, "Dataset still has missing values"

In [8]:
out_path = "../data/processed/smart_waste_dataset_cleaned.csv"
dataset.to_csv(out_path, index=False)